<a href="https://colab.research.google.com/github/Kamilr616/Emotion_sentiment_analysis/blob/main/source/Emotion_Sentiment_Analysis_tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install dependencies

Install compatible Python libraries for the project. The exact 2024 environment used for the archived report is recorded in requirements-recorded.txt.

In [ ]:
%pip install -q "tensorflow>=2.15,<3" "pandas>=2,<3" "scikit-learn>=1.2,<2" "kaggle>=1.6,<2" "seaborn>=0.13,<1" "nltk>=3.8,<4" "numpy>=1.25,<3" "matplotlib>=3.7,<4"

# Import libraries

Import all required libraries for data processing, model building, and evaluation.


In [ ]:
import json
import os
import re
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from google.colab import files
from nltk.corpus import stopwords
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import BatchNormalization, Bidirectional, Dense, Dropout, Embedding, LSTM, SpatialDropout1D
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical

SEED = 42
VOCAB_SIZE = 60000
MAX_LENGTH = 100
tf.keras.utils.set_random_seed(SEED)

print(f"TensorFlow version = {tf.__version__}")

# Upload configuration files

Upload kaggle.json, datasets.txt, and datasets_source.txt. The uploaded credential is never displayed and is removed from the working directory after the Kaggle API is initialized.

In [ ]:
os.makedirs("datasets", exist_ok=True)
uploaded_files = files.upload()
required_files = {"kaggle.json", "datasets.txt", "datasets_source.txt"}
missing_files = required_files.difference(uploaded_files)
if missing_files:
    raise FileNotFoundError(f"Missing uploaded files: {sorted(missing_files)}")
del uploaded_files

# Initialize Kaggle API

Set up the Kaggle API by configuring authentication.


In [ ]:
credential_source = Path("kaggle.json")
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
credential_target = kaggle_dir / "kaggle.json"
shutil.copy2(credential_source, credential_target)
credential_target.chmod(0o600)
credential_source.unlink()

import kaggle

kaggle.api.authenticate()

# Fetch datasets from Kaggle

Define a function to download datasets from Kaggle using the provided configuration.


In [ ]:
def download_kaggle_datasets(dataset_list: str = "datasets_source.txt") -> None:
    """Download every configured Kaggle dataset and fail if any download fails."""
    failed_datasets = []

    try:
        with open(dataset_list, "r", encoding="utf-8") as file:
            for line in file:
                dataset_ref = line.strip()
                if not dataset_ref or dataset_ref.startswith("#"):
                    continue

                try:
                    kaggle.api.dataset_download_files(
                        dataset_ref, path="datasets", unzip=True
                    )
                    print(f"Successfully downloaded {dataset_ref}")
                except Exception as exc:
                    failed_datasets.append((dataset_ref, str(exc)))
    except FileNotFoundError as exc:
        raise FileNotFoundError(f"File {dataset_list} not found.") from exc

    if failed_datasets:
        raise RuntimeError(f"Failed Kaggle downloads: {failed_datasets}")


download_kaggle_datasets()

#Fetch Stopwords

In [ ]:
nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))

# Load and preprocess datasets

Load the downloaded datasets and perform initial preprocessing.




In [ ]:
def load_datasets_from_file(filename: str = "datasets.txt") -> list[pd.DataFrame]:
    """Load every configured CSV and fail fast on missing or invalid input."""
    dataframes = []

    try:
        with open(filename, "r", encoding="utf-8") as file:
            for line in file:
                dataset_name = line.strip()
                if not dataset_name or dataset_name.startswith("#"):
                    continue

                dataset_path = Path("datasets") / dataset_name
                try:
                    dataframes.append(pd.read_csv(dataset_path))
                except Exception as exc:
                    raise RuntimeError(
                        f"Failed to load {dataset_path}: {exc}"
                    ) from exc
    except FileNotFoundError as exc:
        raise FileNotFoundError(f"File {filename} not found.") from exc

    if not dataframes:
        raise RuntimeError(f"No datasets were loaded from {filename}.")

    return dataframes


df_loaded_list = load_datasets_from_file()
df_loaded_list

# Standardize column names

In [ ]:
def standardize_column_names(
    frame: pd.DataFrame, column_map: dict[str, list[str]]
) -> pd.DataFrame:
    frame = frame.copy()
    frame.columns = [str(column).strip().lower() for column in frame.columns]

    reverse_mapping = {
        synonym: standard
        for standard, synonyms in column_map.items()
        for synonym in synonyms
    }

    for old_name, new_name in reverse_mapping.items():
        if old_name in frame.columns:
            frame.rename(columns={old_name: new_name}, inplace=True)

    missing_columns = {"text", "emotion"}.difference(frame.columns)
    if missing_columns:
        raise ValueError(
            f"Required columns missing after standardization: {sorted(missing_columns)}"
        )

    return frame


column_mapping = {
    "text": ["text", "content", "comment"],
    "emotion": ["emotion", "label", "sentiment"],
}

df_std_list = [
    standardize_column_names(frame, column_mapping)
    for frame in df_loaded_list
]
df_std_list

#Standardize labels

In [ ]:
def standardize_emotion_names(dataframe: pd.DataFrame, emotion_dict: dict) -> pd.DataFrame:
    """
    Standardize the names of emotions in a DataFrame.

    Args:
    dataframe (pd.DataFrame): The DataFrame containing emotion labels.
    emotion_dict (dict): A dictionary mapping original emotion names to standardized names.

    Returns:
    pd.DataFrame: The DataFrame with standardized emotion names.
    """
    dataframe = dataframe.copy()
    dataframe['emotion'] = dataframe['emotion'].apply(
        lambda value: value.strip().lower() if isinstance(value, str) else value
    )
    reverse_emotion_dict = {synonym: emotion for emotion, synonyms in emotion_dict.items() for synonym in synonyms}
    dataframe['emotion'] = dataframe['emotion'].replace(reverse_emotion_dict)

    return dataframe


emotions_dictionary = {
    'sadness': ['sad', 'sadness', 'depressed', 0],
    'happiness': ['happy', 'happiness', 'joy', 1],
    'neutral': ['neutral', 'indifferent', 'unbiased'],
    'worry': ['worry', 'anxiety', 'concern'],
    'surprise': ['surprise', 'astonishment', 'shock', 5],
    'love': ['love', 'affection', 'adoration', 2],
    'anger': ['angry', 'rage', 'outrage', 'anger', 3],
    'relief': ['relief', 'ease', 'comfort'],
    'fear': ['fear', 'dread', 'terror', 4],
    'empty': ['empty', 'void', 'hollow'],
    'fun': ['fun', 'joyful', 'amusing'],
    'hate': ['hate', 'detest', 'loathe'],
    'enthusiasm': ['enthusiastic', 'excited', 'eager'],
    'boredom': ['boredom', 'tedium', 'monotony']
}

df_std_list_2 = []

for _df in df_std_list:
    df_std_list_2.append(standardize_emotion_names(_df, emotions_dictionary))

df_std_list_2

#Check and drop NULL records

In [ ]:
df_std_list_3 = []

for frame in df_std_list_2:
    cleaned_frame = frame.dropna(subset=["text", "emotion"]).copy()
    cleaned_frame["text"] = cleaned_frame["text"].astype(str)
    df_std_list_3.append(cleaned_frame)
    print(cleaned_frame.isnull().any())

#Cleaning records

In [ ]:
def clean_text(text: str) -> str:
    """Remove mentions, links, and non-alphanumeric characters."""
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"http\S+|www\.\S+", "", text)
    return re.sub(r"[^A-Za-z0-9\s]+", "", text)


df_clean_list = []

for frame in df_std_list_3:
    cleaned_frame = frame.copy()
    cleaned_frame["text"] = cleaned_frame["text"].apply(clean_text)
    df_clean_list.append(cleaned_frame)

df_clean_list

# Chat words mapping

This function replaces specific words in a given text with their corresponding values from a predefined dictionary, chat_words.

In [ ]:
def replace_chat_words(text: str) -> str:
    """
    Replace specific words in the input text with their corresponding values from the chat_words dictionary.

    Args:
    text (str): The input text containing words to be replaced.

    Returns:
    str: The text with words replaced according to the chat_words dictionary.
    """

    words = text.split()
    for i, word in enumerate(words):
        key = word.upper()
        if key in chat_words:
            words[i] = chat_words[key]
    return ' '.join(words)


chat_words = {
    "AFAIK": "As Far As I Know",
    "AFK": "Away From Keyboard",
    "ASAP": "As Soon As Possible",
    "ATK": "At The Keyboard",
    "ATM": "At The Moment",
    "A3": "Anytime, Anywhere, Anyplace",
    "BAK": "Back At Keyboard",
    "BBL": "Be Back Later",
    "BBS": "Be Back Soon",
    "BFN": "Bye For Now",
    "B4N": "Bye For Now",
    "BRB": "Be Right Back",
    "BRT": "Be Right There",
    "BTW": "By The Way",
    "B4": "Before",
    "B4N": "Bye For Now",
    "CU": "See You",
    "CUL8R": "See You Later",
    "CYA": "See You",
    "FAQ": "Frequently Asked Questions",
    "FC": "Fingers Crossed",
    "FWIW": "For What It's Worth",
    "FYI": "For Your Information",
    "GAL": "Get A Life",
    "GG": "Good Game",
    "GN": "Good Night",
    "GMTA": "Great Minds Think Alike",
    "GR8": "Great!",
    "G9": "Genius",
    "IC": "I See",
    "ICQ": "I Seek you (also a chat program)",
    "ILU": "ILU: I Love You",
    "IMHO": "In My Honest/Humble Opinion",
    "IMO": "In My Opinion",
    "IOW": "In Other Words",
    "IRL": "In Real Life",
    "KISS": "Keep It Simple, Stupid",
    "LDR": "Long Distance Relationship",
    "LMAO": "Laugh My A.. Off",
    "LOL": "Laughing Out Loud",
    "LTNS": "Long Time No See",
    "L8R": "Later",
    "MTE": "My Thoughts Exactly",
    "M8": "Mate",
    "NRN": "No Reply Necessary",
    "OIC": "Oh I See",
    "PITA": "Pain In The A..",
    "PRT": "Party",
    "PRW": "Parents Are Watching",
    "QPSA?": "Que Pasa?",
    "ROFL": "Rolling On The Floor Laughing",
    "ROFLOL": "Rolling On The Floor Laughing Out Loud",
    "ROTFLMAO": "Rolling On The Floor Laughing My A.. Off",
    "SK8": "Skate",
    "STATS": "Your sex and age",
    "ASL": "Age, Sex, Location",
    "THX": "Thank You",
    "TTFN": "Ta-Ta For Now!",
    "TTYL": "Talk To You Later",
    "U": "You",
    "U2": "You Too",
    "U4E": "Yours For Ever",
    "WB": "Welcome Back",
    "WTF": "What The F...",
    "WTG": "Way To Go!",
    "WUF": "Where Are You From?",
    "W8": "Wait...",
    "7K": "Sick:-D Laugher",
    "TFW": "That feeling when",
    "MFW": "My face when",
    "MRW": "My reaction when",
    "IFYP": "I feel your pain",
    "TNTL": "Trying not to laugh",
    "JK": "Just kidding",
    "IDC": "I don't care",
    "ILY": "I love you",
    "IMU": "I miss you",
    "ADIH": "Another day in hell",
    "ZZZ": "Sleeping, bored, tired",
    "WYWH": "Wish you were here",
    "TIME": "Tears in my eyes",
    "BAE": "Before anyone else",
    "FIMH": "Forever in my heart",
    "BSAAW": "Big smile and a wink",
    "BWL": "Bursting with laughter",
    "BFF": "Best friends forever",
    "CSL": "Can't stop laughing"
}

df_clean_list_2 = []

for frame in df_clean_list:
    expanded_frame = frame.copy()
    expanded_frame['text'] = expanded_frame['text'].apply(replace_chat_words)
    df_clean_list_2.append(expanded_frame)

df_clean_list_2

# Stopwords

In [ ]:
def remove_stop_words_from_dfs(
    df_list: list[pd.DataFrame], stop_words: set[str]
) -> list[pd.DataFrame]:
    """Remove English stop words case-insensitively."""
    processed_frames = []

    for frame in df_list:
        processed_frame = frame.copy()
        processed_frame["text"] = processed_frame["text"].apply(
            lambda text: " ".join(
                word for word in text.split() if word.lower() not in stop_words
            )
        )
        processed_frames.append(processed_frame)

    return processed_frames


df_words_list = remove_stop_words_from_dfs(df_clean_list_2, stop_words)
df_words_list

# Normalize whitespace

Normalize repeated whitespace and trim leading or trailing spaces.

In [ ]:
def normalize_whitespace_in_dfs(
    df_list: list[pd.DataFrame],
) -> list[pd.DataFrame]:
    """Normalize whitespace in every text column."""
    normalized_frames = []

    for frame in df_list:
        normalized_frame = frame.copy()
        normalized_frame["text"] = (
            normalized_frame["text"]
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )
        normalized_frames.append(normalized_frame)

    return normalized_frames


df_normalized_list = normalize_whitespace_in_dfs(df_words_list)
df_normalized_list

# Merge dataframes

In [ ]:
def concat_dataframes(dataframes: list[pd.DataFrame]) -> pd.DataFrame:
    """Concatenate standardized text and emotion columns."""
    return pd.concat(
        [frame[["text", "emotion"]] for frame in dataframes],
        ignore_index=True,
    )


df_merged = concat_dataframes(df_normalized_list)
df_merged = df_merged[df_merged["text"].str.len().gt(0)].copy()
unexpected_emotions = sorted(
    set(df_merged["emotion"]).difference(emotions_dictionary),
    key=str,
)
if unexpected_emotions:
    raise ValueError(f"Unexpected emotion labels: {unexpected_emotions}")

df_merged

# Drop duplicates

This function removes duplicate rows from a dataframe based on the 'text' column and returns the dataframe with unique records. Additionally, it calculates and prints the number of unique records.

In [ ]:
def remove_duplicates_and_count(df: pd.DataFrame) -> pd.DataFrame:
    """Remove duplicate texts and return an independent DataFrame."""
    return df.drop_duplicates(subset=["text"]).copy()


df_unique = remove_duplicates_and_count(df_merged)
print(f"Number of unique records: {len(df_unique)}")
df_unique

# Keep model columns

In [ ]:
df_unique = df_unique[["text", "emotion"]].copy()
df_unique.shape

# Shuffle records reproducibly

In [ ]:
df_shuffled = df_unique.sample(frac=1, random_state=SEED).reset_index(drop=True)
df_shuffled

# Describe and check dataset

In [ ]:
df_shuffled.describe()

#Visualize content

In [ ]:
df_shuffled['emotion'].value_counts()

In [ ]:
plt.figure(figsize=(12, 8))
sns.countplot(x='emotion', data=df_shuffled)
plt.title('Emotion Visualization')
plt.show()

# Label encoding and stratified train/validation/test split

In [ ]:
texts = df_shuffled["text"].astype(str).to_numpy()
labels = df_shuffled["emotion"].to_numpy()

label_encoder = LabelEncoder()
label_indices = label_encoder.fit_transform(labels)
num_classes = len(label_encoder.classes_)

X_train_texts, X_holdout_texts, y_train_idx, y_holdout_idx = train_test_split(
    texts,
    label_indices,
    test_size=0.2,
    random_state=SEED,
    stratify=label_indices,
)
X_val_texts, X_test_texts, y_val_idx, y_test_idx = train_test_split(
    X_holdout_texts,
    y_holdout_idx,
    test_size=0.5,
    random_state=SEED,
    stratify=y_holdout_idx,
)

y_train = to_categorical(y_train_idx, num_classes=num_classes)
y_val = to_categorical(y_val_idx, num_classes=num_classes)
y_test = to_categorical(y_test_idx, num_classes=num_classes)

# Tokenization fitted on training data only

In [ ]:
tokenizer = Tokenizer(
    num_words=VOCAB_SIZE, lower=True, oov_token="<OOV>"
)
tokenizer.fit_on_texts(X_train_texts)
word_index = tokenizer.word_index

X_train = pad_sequences(
    tokenizer.texts_to_sequences(X_train_texts), maxlen=MAX_LENGTH
)
X_val = pad_sequences(
    tokenizer.texts_to_sequences(X_val_texts), maxlen=MAX_LENGTH
)
X_test = pad_sequences(
    tokenizer.texts_to_sequences(X_test_texts), maxlen=MAX_LENGTH
)

# Split summary and class weights

Confirm the independent 80/10/10 split and compute balanced class weights from the training labels only.

In [ ]:
class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=y_train_idx,
)
class_weights = dict(enumerate(class_weight_values))

print(
    f"Train: {len(X_train):,}; validation: {len(X_val):,}; "
    f"test: {len(X_test):,}"
)
print(f"Classes: {label_encoder.classes_.tolist()}")

# Build the model

Construct the neural network model using Keras.


In [ ]:
effective_vocab_size = min(VOCAB_SIZE, len(word_index) + 1)

model = Sequential(
    [
        Embedding(input_dim=effective_vocab_size, output_dim=100),
        SpatialDropout1D(0.2),
        Bidirectional(LSTM(128)),
        BatchNormalization(),
        Dropout(0.5),
        Dense(num_classes, activation="softmax"),
    ]
)

model.compile(
    loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"]
)
model.summary()

# Train and evaluate model

Train with a dedicated validation split and balanced class weights, then evaluate once on the independent test split.

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss", patience=2, restore_best_weights=True
)
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_val, y_val),
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=2,
)

loss, accuracy = model.evaluate(X_test, y_test, verbose=2)
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Loss: {loss:.4f}")

# Classification report

Generate a detailed classification report to assess the model's performance.

In [ ]:
y_pred = model.predict(X_test)
y_pred_labels = np.argmax(y_pred, axis=1)

print("Classification Report:")
print(
    classification_report(
        y_test_idx,
        y_pred_labels,
        labels=np.arange(num_classes),
        target_names=label_encoder.classes_,
        zero_division=0,
    )
)

# Save inference artifacts

Save the model together with the tokenizer, class order, and preprocessing configuration required for standalone inference.

In [ ]:
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

model.save(artifacts_dir / "emotion_classification_model.keras")
(artifacts_dir / "tokenizer.json").write_text(
    tokenizer.to_json(), encoding="utf-8"
)
(artifacts_dir / "labels.json").write_text(
    json.dumps(label_encoder.classes_.tolist(), indent=2),
    encoding="utf-8",
)
(artifacts_dir / "preprocessing_config.json").write_text(
    json.dumps(
        {
            "vocab_size": VOCAB_SIZE,
            "max_length": MAX_LENGTH,
            "seed": SEED,
        },
        indent=2,
    ),
    encoding="utf-8",
)
print(f"Inference artifacts saved to {artifacts_dir.resolve()}")

# Test model on input data

Evaluate the model on new input texts to predict their emotion categories.

In [ ]:
new_texts = ["The sun is shining and everything seems possible today! I am so uplifted!",
             "Excited to start my new job next week!",
             "Watching the sunset with my loved ones, life is good.",
             "The project deadline is approaching, and I haven't started."]


new_sequences = tokenizer.texts_to_sequences(new_texts)
new_padded_sequences = pad_sequences(new_sequences, maxlen=MAX_LENGTH)
predictions = model.predict(new_padded_sequences)

for i, prediction in enumerate(predictions):
    predicted_label = label_encoder.inverse_transform([prediction.argmax()])[0]
    print(f'Text: {new_texts[i]} - Predicted Emotion: {predicted_label}')